# 🚀 Advanced PySpark Performance Tuning & Optimization (Part 5)
This notebook covers the remaining advanced optimization techniques for enterprise PySpark pipelines: Dynamic Partition Pruning (DPP), Avoiding Cartesian/Broadcast Nested Loop Joins, AQE Dynamic Join Selection, GC Pause & Driver Memory Leak Prevention, and Optimized Higher-Order Array Functions.

--- 
## 17. Dynamic Partition Pruning (DPP)
### **Concept & Architectural Explanation**
* **The Problem:** In star-schema joins between a partitioned fact table and a filtered dimension table, static partition pruning fails if the filter is on the dimension table (e.g., `dim_date.quarter = 'Q3'`). Without DPP, Spark scans *all* partitions of the massive fact table.
* **How DPP Works:** Spark dynamically constructs a runtime filter from the filtered dimension table side and pushes it directly down into the fact table scan via a subquery broadcast filter.
* **Physical Plan Indicator:** Look for `dynamicpruningexpression()` inside the `FileScan` operator.
* **Production Best Practice:** Ensure `spark.sql.optimizer.dynamicPartitionPruning.enabled = true` (enabled by default in Spark 3.x+).

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os
import shutil

spark = SparkSession.builder \
    .appName("Optimization_17_Dynamic_Partition_Pruning") \
    .config("spark.sql.optimizer.dynamicPartitionPruning.enabled", "true") \
    .master("local[*]") \
    .getOrCreate()

fact_path = "/tmp/dpp_fact_sales"
if os.path.exists(fact_path):
    shutil.rmtree(fact_path)

# Step 1: Create a partitioned Fact Table (partitioned by store_id)
fact_data = spark.range(0, 500000).withColumn("store_id", (F.col("id") % 10).cast("int")) \
                                .withColumn("sales_amount", F.rand() * 100)
fact_data.write.partitionBy("store_id").mode("overwrite").parquet(fact_path)

# Step 2: Create a Dimension Table
dim_stores = spark.createDataFrame([
    (1, "Store_East", "Region_A"),
    (2, "Store_West", "Region_B")
], ["store_id", "store_name", "region"])

# Step 3: Join partitioned fact table with filtered dimension table
fact_df = spark.read.parquet(fact_path)
dpp_joined = fact_df.join(F.broadcast(dim_stores), "store_id").filter("region = 'Region_A'")

print("=== Physical Plan: Dynamic Partition Pruning (DPP) ===")
dpp_joined.explain(True)

--- 
## 18. Eliminating Cartesian Products & BroadcastNestedLoopJoin
### **Concept & Architectural Explanation**
* **The Danger:** Non-equi joins (e.g., `df1.join(df2, df1.val > df2.val)`) or missing join keys force Spark into a **`BroadcastNestedLoopJoin` (BNLJ)** or **`CartesianProduct`**, producing $O(M \times N)$ complexity that locks up cluster tasks indefinitely.
* **The Fix:** Rewrite range joins using interval mapping, bucketed window functions, or explode logic to enforce strict equality conditions ($=$) where possible.
* **Production Safety Net:** Keep `spark.sql.crossJoin.enabled = false` in production to throw an explicit error rather than silently executing accidental cross joins.

In [ ]:
# Step 1: Demonstrate detection of Cartesian Product in execution plan
df_a = spark.range(0, 1000).withColumn("val_a", F.col("id"))
df_b = spark.range(0, 1000).withColumn("val_b", F.col("id"))

# Non-equi join triggering high-cost plan
non_equi_join = df_a.join(df_b, df_a.val_a < df_b.val_b)
print("=== Physical Plan: Identifying BroadcastNestedLoopJoin / CartesianProduct ===")
non_equi_join.explain(True)

--- 
## 19. AQE Dynamic Join Selection (Runtime Plan Conversion)
### **Concept & Architectural Explanation**
* **How It Works:** Static Catalyst planning estimates table sizes before execution. If data statistics are stale or filtered data size drops unexpectedly, static planning might choose an expensive Sort-Merge Join.
* **AQE Advantage:** Adaptive Query Execution inspects actual shuffle map stage outputs at runtime. If the filtered shuffle output size drops below `spark.sql.autoBroadcastJoinThreshold`, AQE dynamically converts a Sort-Merge Join into a Broadcast Hash Join at runtime.
* **Physical Plan Indicator:** `AdaptiveSparkPlan isFinalPlan=true` displaying a dynamic switch to `BroadcastHashJoin`.

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", "10485760") # 10MB

# Simulate filtered join where final output is small enough for dynamic broadcast
large_table = spark.range(0, 1000000).withColumn("join_key", F.col("id") % 100)
dim_table = spark.range(0, 100).withColumn("join_key", F.col("id")).filter("id < 5") # Aggressively filtered

aqe_dynamic_join = large_table.join(dim_table, "join_key")
count = aqe_dynamic_join.count() # Action triggers runtime execution plan

print("=== Post-Execution Physical Plan: AQE Dynamic Join Selection ===")
aqe_dynamic_join.explain(True)

--- 
## 20. Driver Memory Leak & GC Pause Mitigation
### **Concept & Architectural Explanation**
* **The Driver Bottleneck:** Accumulating large datasets on the driver process via `.collect()`, `.toPandas()`, or maintaining unbounded driver object references triggers driver Out-Of-Memory (OOM) crashes and severe Garbage Collection (GC) pauses.
* **Production Best Practices:**
  1. Never call `.collect()` on un-aggregated production DataFrames. Use `.take(n)` or `.foreachPartition()` to process partitions in-place.
  2. Clear cached DataFrames explicitly using `.unpersist()` when execution completes.
  3. Configure G1GC in `spark.driver.extraJavaOptions` and `spark.executor.extraJavaOptions`: `-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35`.

In [ ]:
# Demonstration: Safe partition processing without driver memory accumulation
df_stream = spark.range(0, 500000).withColumn("val", F.rand())

# BAD PRACTICE (Avoid in production):
# driver_data = df_stream.collect() # Pulls all rows into JVM driver memory

# GOOD PRACTICE: Process partitions in worker memory without pulling to driver
def process_partition(rows):
    count = 0
    for row in rows:
        count += 1
    print(f"Partition processed {count} rows in worker node.")

df_stream.foreachPartition(process_partition)
print("Partition-level processing completed safely without driver memory overhead.")

spark.stop()